In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_process_step_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalProcessStepId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	    string,"
        #     "LegalProcessStepKey	string,"
        #     "LegalProcessKey	int,"
        #     "LegalStageKey int,"
        #     "LegalStepKey int,"
        #     "LegalActionKey int,"
        #     "ExposureKey string,"
        #     "PartyKey int,"
        #     "CDI string,"
        #     "CaseCode int,"
        #     "LegalStepCreatedBy string,"
        #     "LegalStepCreatedDate timestamp,"
        #     "LegalStepStartDate string,"
        #     "LegalStepEndDate string,"
        #     "LegalStepSystemEndDate string,"
        #     "LegalStepDefinitionCode int,"
        #     "LegalStepDefinition string,"
        #     "LegalStepStatusCode int,"
        #     "LegalStepStatus string,"
        #     "LegalStepComment string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalProcessStep"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Portfolio ==================

        portfolio_extra_filters = [
            "PortfolioName in ('Core Retail','Core Wholesale')"
        ]

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "portfolio_id_qca"]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== fs_cases_snap ==================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = ["cust_code", "CASE_CODE", "SPET_CODE"]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "CASE_CODE",
            "cust_code",
            "debt_code",
            "CASE_ACCOUNT_FOR_SEARCH",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "CUST_CIF",
            "CUST_ID",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ================== DIM_PROCESS_STEP_DEFINITION ==================

        DIM_PROCESS_STEP_DEFINITION_extra_filters = None

        if DIM_PROCESS_STEP_DEFINITION_extra_filters is not None:
            DIM_PROCESS_STEP_DEFINITION_extra_filters.append(date_filter)
            DIM_PROCESS_STEP_DEFINITION_final_filters = " AND ".join(
                DIM_PROCESS_STEP_DEFINITION_extra_filters
            )
        else:
            DIM_PROCESS_STEP_DEFINITION_final_filters = date_filter

        DIM_PROCESS_STEP_DEFINITION_selected_fields = [
            "PDSP_DESC",
            "PDSP_ID",
        ]

        DIM_PROCESS_STEP_DEFINITION = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_DIM_PROCESS_STEP_DEFINITION",
            DIM_PROCESS_STEP_DEFINITION_final_filters,
            DIM_PROCESS_STEP_DEFINITION_selected_fields,
        )

        # ================== FS_PROCESS_INSTANCE_STEPS ==================

        FS_PROCESS_INSTANCE_STEPS_extra_filters = None

        if FS_PROCESS_INSTANCE_STEPS_extra_filters is not None:
            FS_PROCESS_INSTANCE_STEPS_extra_filters.append(date_filter)
            FS_PROCESS_INSTANCE_STEPS_final_filters = " AND ".join(
                FS_PROCESS_INSTANCE_STEPS_extra_filters
            )
        else:
            FS_PROCESS_INSTANCE_STEPS_final_filters = date_filter

        FS_PROCESS_INSTANCE_STEPS_selected_fields = [
            "PD_DESC",
            "PDSP_ID",
            "PDST_ID",
            "PRIN_ID",
            "PROC_STEP_STATUS_ID",
            "PRST_CREATED_BY",
            "PRST_ID",
            "WKFV_COMPL_ID",
            "WKFV_EST_COMPL_ID",
            "WKFV_START_ID",
            "businessdate",
        ]

        FS_PROCESS_INSTANCE_STEPS = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_PROCESS_INSTANCE_STEPS",
            FS_PROCESS_INSTANCE_STEPS_final_filters,
            FS_PROCESS_INSTANCE_STEPS_selected_fields,
        )

        # ================== FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS ==================

        FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_extra_filters = None

        if FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_extra_filters is not None:
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_extra_filters.append(date_filter)
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_final_filters = " AND ".join(
                FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_extra_filters
            )
        else:
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_final_filters = date_filter

        FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_selected_fields = [
            "CASE_CODE",
            "PRST_ID",
        ]

        FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS",
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_final_filters,
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_selected_fields,
        )

        FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct = (
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS.select(
                "CASE_CODE", "PRST_ID"
            ).distinct()
        )

        # ================== Join conditions ==================

        # Join Conditions
        FS_PROCESS_INSTANCE_STEPS_DIM_PROCESS_STEP_DEFINITION_join_condition = (
            FS_PROCESS_INSTANCE_STEPS["PDSP_ID"]
            == DIM_PROCESS_STEP_DEFINITION["PDSP_ID"]
        )

        FS_PROCESS_INSTANCE_STEPS_FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct_join_condition = (
            FS_PROCESS_INSTANCE_STEPS["PRST_ID"]
            == FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct["PRST_ID"]
        )

        FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct_dim_cases_static_join_condition = (
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct["CASE_CODE"]
            == dim_cases_static["CASE_CODE"]
        )

        fs_cases_snap_dim_cases_static_join_condition = (
            fs_cases_snap["CASE_CODE"] == dim_cases_static["CASE_CODE"]
        )

        dim_customer_static_fs_cases_snap_join_condition = (
            dim_customer_static["CUST_ID"] == fs_cases_snap["cust_code"]
        )

        portfolio_fs_cases_snap_join_condition = (
            portfolio.portfolio_id_qca == fs_cases_snap.SPET_CODE
        )

        # Joins
        target_table_qca = (
            FS_PROCESS_INSTANCE_STEPS.join(
                DIM_PROCESS_STEP_DEFINITION,
                FS_PROCESS_INSTANCE_STEPS_DIM_PROCESS_STEP_DEFINITION_join_condition,
                "inner",
            )
            .join(
                FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct,
                FS_PROCESS_INSTANCE_STEPS_FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct_join_condition,
                "inner",
            )
            .join(
                dim_cases_static,
                FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct_dim_cases_static_join_condition,
                "left",
            )
            .join(
                fs_cases_snap,
                fs_cases_snap_dim_cases_static_join_condition,
                "inner",
            )
            .join(
                dim_customer_static,
                dim_customer_static_fs_cases_snap_join_condition,
                "left",
            )
            .join(
                portfolio,
                portfolio_fs_cases_snap_join_condition,
                "inner",
            )
        )

        # HashKeys
        target_table_qca_legal_process_stepId_primary_key_fields = [
            FS_PROCESS_INSTANCE_STEPS.PRST_ID,
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct.CASE_CODE,
            lit("qca"),
        ]

        target_table_qca = (
            target_table_qca.withColumn(
                "LegalProcessStepId",
                sha2(
                    concat_ws(
                        ";", *target_table_qca_legal_process_stepId_primary_key_fields
                    ),
                    256,
                ),
            )
            .withColumn(
                "LegalProcessStepKey",
                concat_ws(
                    ";",
                    FS_PROCESS_INSTANCE_STEPS.PRST_ID,
                    FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct.CASE_CODE,
                ),
            )
            .withColumn(
                "LegalStepStatus",
                when(
                    FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.isin("9112"), "Active"
                )
                .when(
                    FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.isin("9113"),
                    "Cancelled",
                )
                .when(
                    FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.isin("9114"),
                    "Completed",
                )
                .when(
                    FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.isin("9115"),
                    "Pending",
                )
                .when(
                    FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.isin("9116"),
                    "Suspended",
                )
                .when(
                    FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.isin("9368"),
                    "Working",
                )
                .when(
                    FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.isin("13573"),
                    "Postponed",
                )
                .otherwise(None),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_qca = target_table_qca.select(
            target_table_qca.LegalProcessStepId.alias("LegalProcessStepId"),
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_qca.LegalProcessStepKey.alias("LegalProcessStepKey"),
            FS_PROCESS_INSTANCE_STEPS.PRIN_ID.alias("LegalProcessKey"),
            FS_PROCESS_INSTANCE_STEPS.PDST_ID.alias("LegalStageKey"),
            FS_PROCESS_INSTANCE_STEPS.PRST_ID.alias("LegalStepKey"),
            lit(None).alias("LegalActionKey"),
            dim_cases_static.CASE_ACCOUNT_FOR_SEARCH.cast("string").alias(
                "ExposureKey"
            ),
            dim_cases_static.cust_code.alias("PartyKey"),
            dim_customer_static.CUST_CIF.alias("CDI"),
            FSAD_PROCESS_INSTANCE_STEPS_PARTICIPANTS_distinct.CASE_CODE.alias(
                "CaseCode"
            ),
            FS_PROCESS_INSTANCE_STEPS.PRST_CREATED_BY.alias("LegalStepCreatedBy"),
            FS_PROCESS_INSTANCE_STEPS.PD_DESC.alias("LegalStepCreatedDate"),
            FS_PROCESS_INSTANCE_STEPS.WKFV_START_ID.alias("LegalStepStartDate"),
            FS_PROCESS_INSTANCE_STEPS.WKFV_EST_COMPL_ID.alias("LegalStepEndDate"),
            FS_PROCESS_INSTANCE_STEPS.WKFV_COMPL_ID.alias("LegalStepSystemEndDate"),
            FS_PROCESS_INSTANCE_STEPS.PDSP_ID.alias("LegalStepDefinitionCode"),
            DIM_PROCESS_STEP_DEFINITION.PDSP_DESC.alias("LegalStepDefinition"),
            FS_PROCESS_INSTANCE_STEPS.PROC_STEP_STATUS_ID.alias("LegalStepStatusCode"),
            target_table_qca.LegalStepStatus.alias("LegalStepStatus"),
            lit(None).alias("LegalStepComment"),
            FS_PROCESS_INSTANCE_STEPS.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================
        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_legal_process_step_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)